In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from data.fetcher import OptionChainFetcher
from data.cleaner import clean_option_chain
from analytics.arbitrage import (
    check_butterfly_arbitrage,
    check_calendar_arbitrage,
    check_vertical_arbitrage,
    run_all_arbitrage_checks
)
from visualization.heatmaps import plot_arbitrage_heatmap

In [ ]:
# Fetch data
fetcher = OptionChainFetcher('SPY')
chain = fetcher.fetch_current_chain()
clean_chain = clean_option_chain(chain)

print(f"Analyzing {len(clean_chain)} options for arbitrage violations...")

In [ ]:
# Run all checks
results = run_all_arbitrage_checks(clean_chain)

for check_name, result in results.items():
    status = 'PASS' if result.get('pass', True) else 'FAIL'
    violations = result.get('violations', 0)
    print(f"\n{check_name.upper()}:")
    print(f"  Status: {status}")
    print(f"  Violations: {violations}")

In [ ]:
# Butterfly arbitrage check
butterfly_result = check_butterfly_arbitrage(clean_chain)

if butterfly_result.get('violations_df') is not None:
    violations_df = butterfly_result['violations_df']
    if len(violations_df) > 0:
        print(f"Found {len(violations_df)} butterfly violations:")
        display(violations_df.head(10))

In [ ]:
# Calendar spread arbitrage
calendar_result = check_calendar_arbitrage(clean_chain)

if calendar_result.get('violations_df') is not None:
    violations_df = calendar_result['violations_df']
    if len(violations_df) > 0:
        print(f"Found {len(violations_df)} calendar violations:")
        display(violations_df.head(10))

In [ ]:
# Visualize arbitrage violations as heatmap
# Create violation matrix for visualization
moneyness = np.linspace(0.85, 1.15, 20)
expiries = np.linspace(14, 90, 10)

# Placeholder - actual implementation would compute violations per cell
violation_matrix = np.random.rand(len(expiries), len(moneyness)) * 0.1

fig = plot_arbitrage_heatmap(violation_matrix, moneyness, expiries)
fig.show()